##Imports & Downloads

In [ ]:
!pip uninstall -y jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt
!pip install -q --upgrade pip
!pip install -q --upgrade "jax[cuda12]"
!pip install -q --upgrade dm-haiku optax jmp

In [ ]:

import math
import functools
import itertools
from functools import partial
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
import haiku as hk
import optax

jax.config.update("jax_enable_x64", True)

print("JAX version:", jax.__version__)
print("Devices:", jax.devices())

## Neural SDE

In [ ]:
# ============================ Exact 10D Brownian setup ============================
from dataclasses import dataclass
import math
import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
import optax

# Use float32 on Colab GPU/T4 for compatibility and speed.
# Do NOT enable x64 here.
# jax.config.update("jax_enable_x64", True)

print("JAX version:", jax.__version__)
print("Devices:", jax.devices())

@dataclass
class CFG:
    d: int = 10
    T: float = 1.0
    dt: float = 0.01
    n_traj: int = 512
    n_rect: int = 40000
    x_box: float = 2.0

cfg = CFG()

key_main = jax.random.PRNGKey(0)

def simulate_brownian_paths(key, cfg: CFG):
    """
    Exact Brownian paths:
      X_{n+1} = X_n + sqrt(dt) * eps_n
    Returns:
      t: (N+1,)
      x: (n_traj, N+1, d)
    """
    N = int(cfg.T / cfg.dt)
    t = jnp.linspace(0.0, cfg.T, N + 1, dtype=jnp.float64)

    eps = jax.random.normal(key, (cfg.n_traj, N, cfg.d), dtype=jnp.float64)
    dW = jnp.sqrt(jnp.asarray(cfg.dt, dtype=jnp.float64)) * eps

    x0 = jnp.zeros((cfg.n_traj, 1, cfg.d), dtype=jnp.float64)
    x_increments = jnp.cumsum(dW, axis=1)
    x = jnp.concatenate([x0, x_increments], axis=1)
    return t, x

def make_TX_gen_uniform(key, cfg: CFG):
    """
    Uniform point cloud in [0,T] x [-x_box,x_box]^d.
    Returns:
      TX_gen: (B, 1+d) = [t, x1, ..., xd]
    """
    key_t, key_x = jax.random.split(key, 2)

    t = jax.random.uniform(
        key_t,
        (cfg.n_rect, 1),
        minval=jnp.float64(0.0),
        maxval=jnp.float64(cfg.T),
        dtype=jnp.float64,
    )
    x = jax.random.uniform(
        key_x,
        (cfg.n_rect, cfg.d),
        minval=jnp.float64(-cfg.x_box),
        maxval=jnp.float64(cfg.x_box),
        dtype=jnp.float64,
    )
    return jnp.concatenate([t, x], axis=1)

key_main, k1, k2 = jax.random.split(key_main, 3)
t_surr, x_surr = simulate_brownian_paths(k1, cfg)
TX_gen = make_TX_gen_uniform(k2, cfg)

print("t_surr shape:", t_surr.shape, t_surr.dtype)
print("x_surr shape:", x_surr.shape, x_surr.dtype)
print("TX_gen shape:", TX_gen.shape, TX_gen.dtype)

globals().update({
    "cfg": cfg,
    "t_surr": t_surr,
    "x_surr": x_surr,
    "TX_gen": TX_gen,
})

# Optional quick sanity plot: first coordinate of a few Brownian paths
plt.figure(figsize=(6, 4))
n_plot = min(20, cfg.n_traj)
t_plot = np.asarray(jax.device_get(t_surr))
x_plot = np.asarray(jax.device_get(x_surr))
for i in range(n_plot):
    plt.plot(t_plot, x_plot[i, :, 0], alpha=0.6)
plt.xlabel("t")
plt.ylabel("X_t^{(1)}")
plt.title("10D Brownian motion: first coordinate of sample paths")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Generator Neural Net Setup

In [ ]:
# ============================ Non-affine generator setup for 10D Brownian ============================
# Full point generators:
#   X_i = tau_i(t, x) ∂_t + sum_{a=1}^d xi_i^a(t, x) ∂_{x_a}
#
# No affine / linear restriction.

from dataclasses import dataclass
import math
import jax
import jax.numpy as jnp

@dataclass
class GenConfig:
    n_generators: int = 8
    d: int = cfg.d
    hidden_tau: int = 64
    hidden_xi: int = 64
    depth_tau: int = 2
    depth_xi: int = 2
    activation: str = "swish"

gen_cfg = GenConfig()

def glorot(key, fan_in, fan_out):
    lim = math.sqrt(6.0 / (fan_in + fan_out))
    return jax.random.uniform(key, (fan_in, fan_out), minval=-lim, maxval=lim, dtype=jnp.float64)

def init_mlp_params(key, sizes):
    keys = jax.random.split(key, len(sizes) - 1)
    params = []
    for k, (m, n) in zip(keys, zip(sizes[:-1], sizes[1:])):
        params.append({
            "W": glorot(k, m, n),
            "b": jnp.zeros((n,), dtype=jnp.float64),
        })
    return params

def mlp_forward(params, x, activation="tanh"):
    h = x
    for i, layer in enumerate(params):
        W, b = layer["W"], layer["b"]
        h = h @ W + b
        if i < len(params) - 1:
            if activation == "tanh":
                h = jnp.tanh(h)
            elif activation == "relu":
                h = jax.nn.relu(h)
            elif activation == "swish":
                h = jax.nn.silu(h)
            elif activation == "gelu":
                h = jax.nn.gelu(h)
            elif activation == "sin":
                h = jnp.sin(h)
            else:
                raise ValueError(f"Unknown activation {activation}")
    return h

class Normalizer:
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std
    def __call__(self, x):
        return (x - self.mean) / (self.std + 1e-8)

def fit_normalizer(X):
    return Normalizer(jnp.mean(X, axis=0), jnp.std(X, axis=0))

# TX_gen has columns [t, x1, ..., xd]
t_flat_gen = TX_gen[:, 0:1]
tx_norm_gen = fit_normalizer(TX_gen)
t_norm_gen = fit_normalizer(t_flat_gen)

def _sizes(in_dim, hidden, depth, out_dim):
    return [in_dim] + [hidden] * depth + [out_dim]

def tau_forward(params_tau, t_norm, activation="tanh"):
    # t_norm: (..., 1)
    return mlp_forward(params_tau, t_norm, activation=activation)[..., 0:1]

def xi_forward(params_xi, tx_norm, activation="tanh"):
    # tx_norm: (..., 1+d)
    return mlp_forward(params_xi, tx_norm, activation=activation)  # (..., d)

def init_generator_params(key, gen_cfg: GenConfig):
    m = gen_cfg.n_generators
    keys = jax.random.split(key, 2 * m)

    tau_list = []
    xi_list = []

    for i in range(m):
        k_tau = keys[2 * i]
        k_xi = keys[2 * i + 1]

        params_tau = init_mlp_params(
            k_tau,
            _sizes(1, gen_cfg.hidden_tau, gen_cfg.depth_tau, 1),
        )
        params_xi = init_mlp_params(
            k_xi,
            _sizes(1 + gen_cfg.d, gen_cfg.hidden_xi, gen_cfg.depth_xi, gen_cfg.d),
        )

        tau_list.append(params_tau)
        xi_list.append(params_xi)

    return {"tau": tau_list, "xi": xi_list}

key_main, key_gen = jax.random.split(key_main)
params_gen = init_generator_params(key_gen, gen_cfg)

def eval_generators(params_gen, t, x, activation=None):
    """
    t: (B,)
    x: (B,d)
    Returns:
      tau: (m,B)
      xi:  (m,B,d)
    """
    if activation is None:
        activation = gen_cfg.activation

    t = jnp.asarray(t, dtype=jnp.float64)
    x = jnp.asarray(x, dtype=jnp.float64)

    if t.ndim != 1:
        raise ValueError("t must have shape (B,)")
    if x.ndim != 2 or x.shape[1] != gen_cfg.d:
        raise ValueError(f"x must have shape (B,{gen_cfg.d})")

    t_col = t.reshape(-1, 1)                           # (B,1)
    tx = jnp.concatenate([t_col, x], axis=1)          # (B,1+d)

    t_norm = t_norm_gen(t_col)
    tx_norm = tx_norm_gen(tx)

    tau_list = []
    xi_list = []

    for p_tau, p_xi in zip(params_gen["tau"], params_gen["xi"]):
        tau_i = tau_forward(p_tau, t_norm, activation=activation).reshape(-1)   # (B,)
        xi_i = xi_forward(p_xi, tx_norm, activation=activation)                  # (B,d)
        tau_list.append(tau_i)
        xi_list.append(xi_i)

    tau = jnp.stack(tau_list, axis=0)   # (m,B)
    xi = jnp.stack(xi_list, axis=0)     # (m,B,d)
    return tau, xi

eval_generators_jit = jax.jit(eval_generators, static_argnames=("activation",))

print(f"Initialized non-affine generator nets with m={gen_cfg.n_generators}, d={gen_cfg.d}")

globals().update({
    "gen_cfg": gen_cfg,
    "params_gen": params_gen,
    "t_norm_gen": t_norm_gen,
    "tx_norm_gen": tx_norm_gen,
    "tau_forward": tau_forward,
    "xi_forward": xi_forward,
    "eval_generators": eval_generators,
    "eval_generators_jit": eval_generators_jit,
})

In [ ]:
import jax
import jax.numpy as jnp

def eval_generators_tau_xi(params_gen, t, x, *, activation="tanh", normalize_tx=None):
    """
    Returns:
      tau_all: (m, B)
      xi_all:  (m, B, d)
    """
    t = jnp.asarray(t, dtype=jnp.float64)
    x = jnp.asarray(x, dtype=jnp.float64)

    if t.ndim != 1:
        raise ValueError("t must have shape (B,)")
    if x.ndim != 2 or x.shape[1] != gen_cfg.d:
        raise ValueError(f"x must have shape (B,{gen_cfg.d})")

    if normalize_tx is None:
        t_raw, x_raw = t, x
    else:
        t_raw, x_raw = normalize_tx(t, x)
        t_raw = jnp.asarray(t_raw, dtype=jnp.float64)
        x_raw = jnp.asarray(x_raw, dtype=jnp.float64)

    t_col = t_raw.reshape(-1, 1)
    tx_col = jnp.concatenate([t_col, x_raw], axis=1)

    t_norm = t_norm_gen(t_col)
    tx_norm = tx_norm_gen(tx_col)

    taus = []
    xis = []

    for params_tau, params_xi in zip(params_gen["tau"], params_gen["xi"]):
        tau_i = tau_forward(params_tau, t_norm, activation=activation).reshape(-1)
        xi_i = xi_forward(params_xi, tx_norm, activation=activation)
        taus.append(tau_i)
        xis.append(xi_i)

    tau_all = jnp.stack(taus, axis=0)   # (m, B)
    xi_all = jnp.stack(xis, axis=0)     # (m, B, d)
    return tau_all, xi_all

# JIT wrapper (recommended)
eval_generators_tau_xi_jit = jax.jit(eval_generators_tau_xi, static_argnames=("activation", "normalize_tx"))


# Algebraic Losses

## Loss 1

In [ ]:
# ============================ L1 — full non-affine Lie bracket closure + constancy ============================
import jax
import jax.numpy as jnp

def _ordered_pair_indices(n: int):
    pairs = [(i, j) for i in range(n) for j in range(n) if i != j]
    ii = jnp.array([p[0] for p in pairs], dtype=jnp.int32)
    jj = jnp.array([p[1] for p in pairs], dtype=jnp.int32)
    return ii, jj

def make_s1_lie_loss_full(n_generators: int, d: int, rcond: float = 1e-6):
    idx_i, idx_j = _ordered_pair_indices(n_generators)
    reg = jnp.asarray(rcond, dtype=jnp.float64) ** 2

    def _Xi_scalar(params_tau_i, params_xi_i, z):
        z = jnp.asarray(z, dtype=jnp.float64)
        t = z[0:1].reshape(1, 1)             # (1,1)
        x = z[1:].reshape(1, d)              # (1,d)
        tx = jnp.concatenate([t, x], axis=1) # (1,1+d)

        t_norm = t_norm_gen(t)
        tx_norm = tx_norm_gen(tx)

        tau = tau_forward(params_tau_i, t_norm, activation=gen_cfg.activation)[0, 0]
        xi = xi_forward(params_xi_i, tx_norm, activation=gen_cfg.activation)[0]   # (d,)
        return jnp.concatenate([jnp.array([tau], dtype=jnp.float64), xi], axis=0) # (1+d,)

    def _fields_and_jacs(params_gen, z):
        F_list = []
        J_list = []

        for p_tau, p_xi in zip(params_gen["tau"], params_gen["xi"]):
            Xi = lambda zz: _Xi_scalar(p_tau, p_xi, zz)
            Fi = Xi(z)                # (1+d,)
            Ji = jax.jacobian(Xi)(z)  # (1+d,1+d)
            F_list.append(Fi)
            J_list.append(Ji)

        F = jnp.stack(F_list, axis=0)   # (m,1+d)
        J = jnp.stack(J_list, axis=0)   # (m,1+d,1+d)
        return F, J

    def _point_err_and_C(F, J):
        Fi = F[idx_i]   # (K,q)
        Fj = F[idx_j]   # (K,q)
        Ji = J[idx_i]   # (K,q,q)
        Jj = J[idx_j]   # (K,q,q)

        # [X_i, X_j] = J_j F_i - J_i F_j
        B = jnp.einsum("kab,kb->ka", Jj, Fi) - jnp.einsum("kab,kb->ka", Ji, Fj)   # (K,q)

        V = jnp.transpose(F, (1, 0))    # (q,m)
        G = V.T @ V + reg * jnp.eye(n_generators, dtype=jnp.float64)   # (m,m)

        rhs = B @ V                      # (K,m)
        C = jax.vmap(lambda r: jnp.linalg.solve(G, r))(rhs)   # (K,m)
        B_proj = C @ V.T                 # (K,q)
        E = B - B_proj                   # (K,q)

        err = jnp.mean(jnp.sum(E ** 2, axis=1))
        return err, C.T   # (m,K)

    def _loss_impl(params_gen, tx_batch):
        def one_point(z):
            F, J = _fields_and_jacs(params_gen, z)
            return _point_err_and_C(F, J)

        errs, Cs = jax.vmap(one_point)(tx_batch)
        error_sum = jnp.mean(errs)
        var_sum = jnp.mean(jnp.var(Cs, axis=0))
        total = error_sum + var_sum

        aux = {
            "error_sum": error_sum,
            "var_sum": var_sum,
        }
        return total, aux

    return jax.jit(_loss_impl)

s1_lie_loss = make_s1_lie_loss_full(
    n_generators=gen_cfg.n_generators,
    d=cfg.d,
)

## Loss 5 - Functional Independence

In [ ]:
# ============================ L5 — full functional independence over sampled fields ============================
# Build the stacked evaluation matrix A from generator values over a batch:
#   rows = all [tau_i(z_n), xi_i^1(z_n), ..., xi_i^d(z_n)]
#   cols = generators i=1..m
#
# Then penalize near-linear dependence.

import jax
import jax.numpy as jnp

def make_s5_column_independence_loss_full(
    n_generators: int,
    *,
    mode: str = "sigma",
    tau_floor: float = 0.2,
    eps: float = 1e-12,
):
    mode = "sigma" if mode == "sigma" else "corr_l2"
    mode_code = 0 if mode == "sigma" else 1

    def _A_from_batch(params_gen, tx_batch):
        t_batch = tx_batch[:, 0]        # (B,)
        x_batch = tx_batch[:, 1:]       # (B,d)

        tau_vals, xi_vals = eval_generators_jit(params_gen, t_batch, x_batch, activation=gen_cfg.activation)
        # tau_vals: (m,B)
        # xi_vals:  (m,B,d)

        tau_block = tau_vals[:, :, None]                      # (m,B,1)
        comp = jnp.concatenate([tau_block, xi_vals], axis=2) # (m,B,1+d)
        comp = jnp.transpose(comp, (1, 2, 0))                # (B,1+d,m)
        A = comp.reshape(-1, n_generators)                   # ((1+d)B,m)
        return A

    def _loss_impl(params_gen, tx_batch):
        A = _A_from_batch(params_gen, tx_batch)

        col_norms = jnp.linalg.norm(A, axis=0) + eps
        Ahat = A / col_norms[None, :]
        G = Ahat.T @ Ahat

        if mode_code == 0:
            lam = jnp.linalg.eigvalsh(G)
            lam_min = jnp.clip(jnp.min(lam), 0.0, None)
            sigma_min = jnp.sqrt(lam_min)
            loss = jnp.maximum(0.0, jnp.asarray(tau_floor, dtype=G.dtype) - sigma_min)
            aux = {"sigma_min": sigma_min}
        else:
            I = jnp.eye(G.shape[0], dtype=G.dtype)
            off = G - I
            off = off - jnp.diag(jnp.diag(off))
            loss = jnp.sum(off * off)
            aux = {"gram_diag_mean": jnp.mean(jnp.diag(G))}

        return loss, aux

    return jax.jit(_loss_impl)

s5_indep_loss = make_s5_column_independence_loss_full(
    n_generators=gen_cfg.n_generators,
    mode="sigma",
    tau_floor=0.2,
    eps=1e-12,
)

## Loss 6 - SDE Symmetry DE

In [ ]:
# ============================ L6 — projectable Brownian determining-equation loss ============================
# Projectable point generators:
#   X = tau(t) ∂_t + sum_a xi^a(t,x) ∂_{x_a}
#
# Since tau depends only on t by construction, the Brownian determining equations reduce to:
#
#   ∂_{x_b} xi^a = (1/2) tau_t δ_ab
#   xi_t^a + (1/2) Δ_x xi^a = 0
#
# No tau_x penalty is needed anymore.

import jax
import jax.numpy as jnp

def make_s6_brownian_det_loss_projectable(d: int):
    eye_d = jnp.eye(d, dtype=jnp.float64)

    def _tau_scalar(params_tau_i, t_scalar):
        t = jnp.asarray([[t_scalar]], dtype=jnp.float64)   # (1,1)
        t_norm = t_norm_gen(t)
        return tau_forward(params_tau_i, t_norm, activation=gen_cfg.activation)[0, 0]

    def _xi_scalar_component(params_xi_i, a, z):
        """
        z: (1+d,) = [t, x1, ..., xd]
        returns xi^a(t,x)
        """
        z = jnp.asarray(z, dtype=jnp.float64)
        t = z[0:1].reshape(1, 1)        # (1,1)
        x = z[1:].reshape(1, d)         # (1,d)
        tx = jnp.concatenate([t, x], axis=1)   # (1,1+d)
        tx_norm = tx_norm_gen(tx)
        return xi_forward(params_xi_i, tx_norm, activation=gen_cfg.activation)[0, a]

    def _one_generator_residual(p_tau, p_xi, z):
        """
        residual for one generator at point z=(t,x)
        """
        t_scalar = z[0]

        tau_fun_t = lambda tt: _tau_scalar(p_tau, tt)
        tau_t = jax.grad(tau_fun_t)(t_scalar)

        r_xi_grad = 0.0
        r_xi_heat = 0.0

        for a in range(d):
            xi_fun_a = lambda zz, aa=a: _xi_scalar_component(p_xi, aa, zz)

            grad_xi_a = jax.grad(xi_fun_a)(z)     # (1+d,)
            xi_t_a = grad_xi_a[0]
            xi_x_a = grad_xi_a[1:]                # (d,)

            H_xi_a = jax.hessian(xi_fun_a)(z)     # (1+d,1+d)
            lap_xi_a = jnp.trace(H_xi_a[1:, 1:])  # Laplacian in x only

            target_grad = 0.5 * tau_t * eye_d[a]  # (d,)

            r_xi_grad = r_xi_grad + jnp.sum((xi_x_a - target_grad) ** 2)
            r_xi_heat = r_xi_heat + (xi_t_a + 0.5 * lap_xi_a) ** 2

        return r_xi_grad + r_xi_heat

    def _point_residual(params_gen, z):
        vals = []
        for p_tau, p_xi in zip(params_gen["tau"], params_gen["xi"]):
            vals.append(_one_generator_residual(p_tau, p_xi, z))
        vals = jnp.stack(vals, axis=0)   # (m,)
        return jnp.mean(vals), vals

    def _loss_impl(params_gen, tx_batch):
        mean_per_point, per_point_per_gen = jax.vmap(lambda z: _point_residual(params_gen, z))(tx_batch)
        loss = jnp.mean(mean_per_point)
        aux = {
            "per_point": mean_per_point,
            "per_gen": jnp.mean(per_point_per_gen, axis=0),
        }
        return loss, aux

    return jax.jit(_loss_impl)

s6_det_loss = make_s6_brownian_det_loss_projectable(cfg.d)

# Master Loss

In [ ]:
# ============================ Master loss (non-affine Brownian: L1 + L5 + L6) ============================
from dataclasses import dataclass
import jax
import jax.numpy as jnp

@dataclass
class LossWeights:
    w_s1_closure: float = 1.0
    w_s5_indep: float = 0.5
    w_s6_det: float = 10.0
    weight_decay: float = 1e-6

loss_cfg = LossWeights()

def l2_tree(params):
    return sum(jnp.sum(jnp.square(p)) for p in jax.tree_util.tree_leaves(params))

def master_loss(params_gen, tx_batch, key=None):
    total = 0.0
    aux = {}

    loss_s1, aux_s1 = s1_lie_loss(params_gen, tx_batch)
    total = total + loss_cfg.w_s1_closure * loss_s1
    aux["L1"] = {"loss": loss_s1, **aux_s1}

    loss_s5, aux_s5 = s5_indep_loss(params_gen, tx_batch)
    total = total + loss_cfg.w_s5_indep * loss_s5
    aux["L5"] = {"loss": loss_s5, **aux_s5}

    loss_s6, aux_s6 = s6_det_loss(params_gen, tx_batch)
    total = total + loss_cfg.w_s6_det * loss_s6
    aux["L6"] = {"loss": loss_s6, **aux_s6}

    wd = loss_cfg.weight_decay * l2_tree(params_gen)
    total = total + wd
    aux["weight_decay"] = wd
    aux["total"] = total

    return total, aux

master_loss_jit = jax.jit(master_loss)

globals().update({
    "loss_cfg": loss_cfg,
    "master_loss": master_loss,
    "master_loss_jit": master_loss_jit,
})

# Training

In [ ]:
# ============================ Training ============================
from dataclasses import dataclass
import numpy as np
import jax
import jax.numpy as jnp
import optax

@dataclass
class GenTrainConfig:
    steps: int = 2000
    batch_size: int = 256
    lr: float = 1e-4
    print_every: int = 50

gen_train_cfg = GenTrainConfig()

optimizer_gen = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adam(gen_train_cfg.lr),
)

opt_state_gen = optimizer_gen.init(params_gen)

TX_gen_np = np.asarray(TX_gen, dtype=np.float64)
mask = np.isfinite(TX_gen_np).all(axis=1)
TX_gen_np = TX_gen_np[mask]
N_tx = TX_gen_np.shape[0]
assert N_tx > 0, "No finite TX_gen points."

rng_np_gen = np.random.default_rng(42)

def sample_tx_batch(batch_size: int):
    if batch_size >= N_tx:
        idx = np.arange(N_tx)
    else:
        idx = rng_np_gen.choice(N_tx, size=batch_size, replace=False)
    return jnp.asarray(TX_gen_np[idx], dtype=jnp.float64)

@jax.jit
def train_step(params_gen, opt_state_gen, tx_batch):
    def loss_for_grad(p):
        return master_loss(p, tx_batch)

    (loss_val, aux), grads = jax.value_and_grad(loss_for_grad, has_aux=True)(params_gen)
    updates, opt_state_new = optimizer_gen.update(grads, opt_state_gen, params_gen)
    params_new = optax.apply_updates(params_gen, updates)
    return params_new, opt_state_new, loss_val, aux

def _safe_float(x):
    return float(jnp.asarray(x))

loss_history_gen = []
loss_hist = {"L1": [], "L5": [], "L6": []}
wd_history = []

print(f"Starting generator training for {gen_train_cfg.steps} steps "
      f"with batch_size={gen_train_cfg.batch_size}, lr={gen_train_cfg.lr}")

for step in range(1, gen_train_cfg.steps + 1):
    tx_batch = sample_tx_batch(gen_train_cfg.batch_size)

    params_gen, opt_state_gen, loss_val, aux = train_step(params_gen, opt_state_gen, tx_batch)

    loss_history_gen.append(_safe_float(loss_val))
    loss_hist["L1"].append(_safe_float(aux["L1"]["loss"]))
    loss_hist["L5"].append(_safe_float(aux["L5"]["loss"]))
    loss_hist["L6"].append(_safe_float(aux["L6"]["loss"]))
    wd_history.append(_safe_float(aux["weight_decay"]))

    if (step % gen_train_cfg.print_every == 0) or (step == 1) or (step == gen_train_cfg.steps):
        msg = (
            f"step {step:5d}/{gen_train_cfg.steps} | "
            f"total={_safe_float(loss_val):.6e} | "
            f"L1={_safe_float(aux['L1']['loss']):.3e} | "
            f"L5={_safe_float(aux['L5']['loss']):.3e} | "
            f"L6={_safe_float(aux['L6']['loss']):.3e} | "
            f"wd={_safe_float(aux['weight_decay']):.3e}"
        )
        if "sigma_min" in aux["L5"]:
            msg += f" | sigma_min={_safe_float(aux['L5']['sigma_min']):.3e}"
        print(msg)

print("\nGenerator training finished.")

globals().update({
    "params_gen": params_gen,
    "opt_state_gen": opt_state_gen,
    "loss_history_gen": loss_history_gen,
    "loss_hist": loss_hist,
    "wd_history": wd_history,
    "gen_train_cfg": gen_train_cfg,
})